In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Open this notebook from the NutriCore repository or its notebooks directory.")

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
os.chdir(PROJECT_ROOT)

# Recipe Recommendation Engine

## Objective

The objective of this notebook is to build a personalized recipe recommendation engine for the NutriCore.

Previous notebooks focused on:

1. Cleaning and preprocessing recipe data.
2. Preparing review data for future enhancements.
3. Calculating personalized calorie and macronutrient targets.

Using these outputs, the recommendation engine will identify recipes that best match a user's nutritional requirements.

The recommendation process considers:

- Calories
- Protein
- Carbohydrates
- Fat
- Number of meals per day

In addition, serving sizes will be adjusted dynamically to improve nutritional matching between user requirements and recipe nutrition profiles.

The final output of this notebook will be a ranked list of recipes along with recommended serving sizes and nutritional information.

In [2]:
# Imports

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## Load Processed Dataset

The dataset used in this notebook has already been cleaned and prepared during the preprocessing stage.

The dataset contains:

- Recipe information
- Dietary classification
- Nutritional information
- Per-serving nutritional features

These features will be used directly by the recommendation engine.

In [3]:
# Load Dataset

df = pd.read_parquet("data/processed/recipes_processed.parquet")

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (302965, 16)


,RecipeId,Name,Description,RecipeCategory,Keywords,RecipeIngredientParts,Calories,FatContent,CarbohydrateContent,ProteinContent,RecipeServings,DietType,CaloriesPerServing,ProteinPerServing,CarbsPerServing,FatPerServing
0,38.0,Low-Fat Berry Blue Frozen Dessert,Make and share this Low-Fat Berry Blue Frozen ...,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[blueberries, granulated sugar, vanilla yogurt...",170.9,2.5,37.1,3.2,4.0,Vegetarian,42.725000,0.800000,9.275000,0.625
1,39.0,Biryani,Make and share this Biryani recipe from Food.com.,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[saffron, milk, hot green chili peppers, onion...",1110.7,58.8,84.4,63.4,6.0,Non-Vegetarian,185.116667,10.566667,14.066667,9.800
2,40.0,Best Lemonade,This is from one of my first Good House Keepi...,Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer...","[sugar, lemons, rind of, lemon, zest of, fresh...",311.1,0.2,81.5,0.3,4.0,Vegetarian,77.775000,0.075000,20.375000,0.050
3,41.0,Carina's Tofu-Vegetable Kebabs,This dish is best prepared a day in advance to...,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[extra firm tofu, eggplant, zucchini, mushroom...",536.1,24.0,64.2,29.3,2.0,Vegetarian,268.050000,14.650000,32.100000,12.000
4,42.0,Cabbage Soup,Make and share this Cabbage Soup recipe from F...,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[plain tomato juice, cabbage, onion, carrots, ...",103.6,0.4,25.1,4.3,4.0,Vegetarian,25.900000,1.075000,6.275000,0.100


In [4]:
# Verify Required Features

required_cols = [
    "CaloriesPerServing",
    "ProteinPerServing",
    "CarbsPerServing",
    "FatPerServing"
]

print(df[required_cols].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302965 entries, 0 to 302964
Data columns (total 4 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CaloriesPerServing  302965 non-null  float64
 1   ProteinPerServing   302965 non-null  float64
 2   CarbsPerServing     302965 non-null  float64
 3   FatPerServing       302965 non-null  float64
dtypes: float64(4)
memory usage: 9.2 MB
None


## Recommendation Strategy

The objective of the recommendation engine is to identify recipes that best match a user's nutritional requirements.

The system receives:

- Daily calorie target
- Daily protein target
- Daily carbohydrate target
- Daily fat target
- Number of meals consumed per day

Since individual recipes represent meals rather than an entire daily diet, daily nutrition targets must first be converted into per-meal targets.

Each recipe will then be compared against these meal-level nutritional requirements.

Recipes with nutritional profiles closest to the target values will receive higher recommendation scores.

### Example User Targets

Before implementing the recommendation algorithm, an example user profile will be created.

This allows the recommendation workflow to be tested and validated before integrating user inputs through a frontend application.

The values below represent sample daily nutrition targets generated by the Nutrition Target Calculation module.

In [5]:
### Example User Targets

target_calories = 2200
target_protein = 160
target_carbs = 250
target_fat = 65

meals_per_day = 4

print("Daily Targets")
print("-" * 30)
print(f"Calories : {target_calories}")
print(f"Protein  : {target_protein}")
print(f"Carbs    : {target_carbs}")
print(f"Fat      : {target_fat}")
print(f"Meals    : {meals_per_day}")

Daily Targets
------------------------------
Calories : 2200
Protein  : 160
Carbs    : 250
Fat      : 65
Meals    : 4


### Dietary Preference

In addition to nutritional requirements, recipe recommendations must satisfy the user's dietary preference.

The recommendation engine supports:

- Vegetarian
- Eggitarian
- Non-Vegetarian

Recipes are filtered before nutritional matching is performed.

In [6]:
### Dietary Preference

diet_type = "Non-Vegetarian"

print(f"Selected Diet Type : {diet_type}")

Selected Diet Type : Non-Vegetarian


### Candidate Pool

Recipes are filtered according to the selected dietary preference.

The dietary hierarchy is:

Vegetarian ⊂ Eggitarian ⊂ Non-Vegetarian

This ensures recommendations align with user preferences.

In [7]:
# Filter Recipes By Diet Preference

if diet_type == "Vegetarian":

    candidate_df = df[
        df["DietType"] == "Vegetarian"
    ].copy()

elif diet_type == "Eggitarian":

    candidate_df = df[
        df["DietType"].isin([
            "Vegetarian",
            "Eggitarian"
        ])
    ].copy()

else:

    candidate_df = df.copy()

print(f"Available Recipes : {len(candidate_df):,}")

Available Recipes : 302,965


### Meal Targets

The recommendation engine compares recipes against meal-level nutritional requirements rather than daily targets.

To achieve this, daily calories and macronutrients are divided by the number of meals consumed per day.

The resulting values represent the nutritional profile of an ideal meal for the user.

In [8]:
# Calculate Meal Targets

meal_calories = target_calories / meals_per_day
meal_protein = target_protein / meals_per_day
meal_carbs = target_carbs / meals_per_day
meal_fat = target_fat / meals_per_day

meal_targets = pd.DataFrame({
    "Metric": ["Calories", "Protein", "Carbs", "Fat"],
    "Target": [
        meal_calories,
        meal_protein,
        meal_carbs,
        meal_fat
    ]
})

meal_targets

,Metric,Target
0,Calories,550.00
1,Protein,40.00
2,Carbs,62.50
3,Fat,16.25


In [9]:
# Calculate Meal Targets

meal_calories = target_calories / meals_per_day
meal_protein = target_protein / meals_per_day
meal_carbs = target_carbs / meals_per_day
meal_fat = target_fat / meals_per_day

meal_targets = pd.DataFrame({
    "Metric": ["Calories", "Protein", "Carbs", "Fat"],
    "Target": [
        meal_calories,
        meal_protein,
        meal_carbs,
        meal_fat
    ]
})

meal_targets

,Metric,Target
0,Calories,550.00
1,Protein,40.00
2,Carbs,62.50
3,Fat,16.25


### Meal Target Findings

The calculated meal targets represent the nutritional requirements that a recommended recipe should ideally satisfy.

For this example:

- Daily Calories = 2200 kcal
- Daily Protein = 160 g
- Daily Carbohydrates = 250 g
- Daily Fat = 65 g
- Meals Per Day = 4

Therefore, each meal should approximately provide:

- 550 kcal
- 40 g protein
- 62.5 g carbohydrates
- 16.25 g fat

The recommendation engine will attempt to identify recipes that closely match these values.

## Serving Size Optimization

Recipes in the dataset contain nutritional values on a per-serving basis.

However, users may consume more or less than a single serving depending on their nutritional requirements.

For example:

- A recipe may provide 200 kcal and 10 g protein per serving.
- A user's meal target may require 550 kcal and 40 g protein.

In such cases, recommending exactly one serving may not be appropriate.

To improve recommendation quality, multiple serving sizes will be evaluated for every recipe.

To recommend realistic meals, the recommendation engine evaluates multiple serving quantities for each recipe and identifies the serving size that best matches the user's nutritional targets.

The search range is limited to the recipe's original serving count. This prevents very low-calorie recipes from matching meal targets simply by recommending an unrealistic number of servings, while also avoiding unrealistically small portions of recipes intended for larger servings.

The serving size that produces the closest nutritional match to the user's meal targets will be selected.



### Candidate Serving Sizes

The candidate serving sizes are determined using the recipe's original serving count.

For example:

- RecipeServings = 2 → 1, 2
- RecipeServings = 4 → 1, 2, 3, 4
- RecipeServings = 6 → 1, 2, 3, 4, 5, 6

Each candidate serving size is evaluated and the serving quantity producing the closest nutritional match is selected.

In [10]:
# Example Recipe

sample_recipe = candidate_df.iloc[301296]

sample_recipe[
    [
        "Name",
        "RecipeServings",
        "CaloriesPerServing",
        "ProteinPerServing",
        "CarbsPerServing",
        "FatPerServing"
    ]
]

Name                  Spinach Artichoke Mac and Cheese
RecipeServings                                    10.0
CaloriesPerServing                               62.75
ProteinPerServing                                 2.94
CarbsPerServing                                   6.29
FatPerServing                                     2.99
Name: 301296, dtype: object

In [11]:
### Candidate Serving Sizes

serving_sizes = np.arange(
    1,
    int(sample_recipe["RecipeServings"]) + 1
)

serving_sizes

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

### Serving Adjustment Example

Before applying the optimization process to the entire dataset, a single recipe will be examined.

The nutritional values corresponding to each valid serving quantity will be calculated and compared against the target meal requirements.

In [12]:
# Serving Adjustment Example

example_rows = []

for serving in serving_sizes:

    example_rows.append({
        "ServingSize": serving,
        "Calories": sample_recipe["CaloriesPerServing"] * serving,
        "Protein": sample_recipe["ProteinPerServing"] * serving,
        "Carbs": sample_recipe["CarbsPerServing"] * serving,
        "Fat": sample_recipe["FatPerServing"] * serving
    })

example_df = pd.DataFrame(example_rows)

example_df

,ServingSize,Calories,Protein,Carbs,Fat
0,1,62.75,2.94,6.29,2.99
1,2,125.50,5.88,12.58,5.98
2,3,188.25,8.82,18.87,8.97
3,4,251.00,11.76,25.16,11.96
4,5,313.75,14.70,31.45,14.95
5,6,376.50,17.64,37.74,17.94
6,7,439.25,20.58,44.03,20.93
7,8,502.00,23.52,50.32,23.92
8,9,564.75,26.46,56.61,26.91
9,10,627.50,29.40,62.90,29.90


### Serving Optimization Findings

As the number of servings increases, the nutritional values increase proportionally.

The recommendation engine evaluates all valid serving quantities and selects the serving size that provides the closest match to the target meal requirements.

## Distance-Based Recipe Matching

After determining the optimal serving size for a recipe, the nutritional profile of the adjusted recipe must be compared against the user's meal targets.

A distance-based scoring approach will be used.

The recommendation engine evaluates:

- Calories
- Protein
- Carbohydrates
- Fat

Because these nutritional components have different scales, raw differences cannot be compared directly.

For example:

- A calorie difference of 50 kcal
- A protein difference of 5 g

do not have the same meaning numerically.

To ensure fair comparison, differences will be normalized relative to the target values.

Lower distance values indicate better nutritional matches.

### Distance Score Weights

The recommendation engine uses a weighted nutritional distance score.

Protein receives the highest weight because protein intake is often the most important nutritional consideration for fitness-related goals.

These weights can be adjusted during experimentation.

In [13]:
### Distance Score Weights

CALORIE_WEIGHT = 0.30
PROTEIN_WEIGHT = 0.40
CARB_WEIGHT = 0.15
FAT_WEIGHT = 0.15

### Distance Calculation Example

Before applying the scoring algorithm to the full dataset, a single serving-size example will be evaluated.

This helps verify that the distance calculation behaves as expected.

In [14]:
### Distance Calculation Example

example_df["CalorieDiff"] = (
    abs(example_df["Calories"] - meal_calories)
    / meal_calories
)

example_df["ProteinDiff"] = (
    abs(example_df["Protein"] - meal_protein)
    / meal_protein
)

example_df["CarbDiff"] = (
    abs(example_df["Carbs"] - meal_carbs)
    / meal_carbs
)

example_df["FatDiff"] = (
    abs(example_df["Fat"] - meal_fat)
    / meal_fat
)

example_df["Distance"] = (
    CALORIE_WEIGHT * example_df["CalorieDiff"]
    + PROTEIN_WEIGHT * example_df["ProteinDiff"]
    + CARB_WEIGHT * example_df["CarbDiff"]
    + FAT_WEIGHT * example_df["FatDiff"]
)


example_df.round(4)

,ServingSize,Calories,Protein,Carbs,Fat,CalorieDiff,ProteinDiff,CarbDiff,FatDiff,Distance
0,1,62.75,2.94,6.29,2.99,0.8859,0.9265,0.8994,0.816,0.8937
1,2,125.50,5.88,12.58,5.98,0.7718,0.8530,0.7987,0.632,0.7874
2,3,188.25,8.82,18.87,8.97,0.6577,0.7795,0.6981,0.448,0.6810
3,4,251.00,11.76,25.16,11.96,0.5436,0.7060,0.5974,0.264,0.5747
4,5,313.75,14.70,31.45,14.95,0.4295,0.6325,0.4968,0.080,0.4684
5,6,376.50,17.64,37.74,17.94,0.3155,0.5590,0.3962,0.104,0.3933
6,7,439.25,20.58,44.03,20.93,0.2014,0.4855,0.2955,0.288,0.3421
7,8,502.00,23.52,50.32,23.92,0.0873,0.4120,0.1949,0.472,0.2910
8,9,564.75,26.46,56.61,26.91,0.0268,0.3385,0.0942,0.656,0.2560
9,10,627.50,29.40,62.90,29.90,0.1409,0.2650,0.0064,0.840,0.2752


In [15]:
# Best Serving Size

best_match = example_df.loc[
    example_df["Distance"].idxmin()
]

best_match

ServingSize      9.000000
Calories       564.750000
Protein         26.460000
Carbs           56.610000
Fat             26.910000
CalorieDiff      0.026818
ProteinDiff      0.338500
CarbDiff         0.094240
FatDiff          0.656000
Distance         0.255981
Name: 8, dtype: float64

## Recommendation Evaluation

Before applying the recommendation algorithm to the entire dataset, a subset of recipes will be evaluated.

For each recipe:

1. Multiple serving sizes will be evaluated.
2. A weighted nutritional distance score will be calculated.
3. The serving size producing the lowest distance will be selected.
4. Recipes will be ranked according to their minimum distance scores.

This experiment validates the recommendation workflow before scaling to the complete dataset.

In [16]:
# Sample Dataset

sample_df = candidate_df.sample(
    n=100,
    random_state=42
)

print(f"Sample Recipes : {len(sample_df)}")

Sample Recipes : 100


In [17]:
# Recommendation Engine Test

recommendations = []

for _, row in sample_df.iterrows():

    serving_sizes = np.arange(
        1,
        int(row["RecipeServings"]) + 1
    )

    best_distance = np.inf
    best_serving = None

    best_calories = None
    best_protein = None
    best_carbs = None
    best_fat = None

    for serving in serving_sizes:

        calories = row["CaloriesPerServing"] * serving
        protein = row["ProteinPerServing"] * serving
        carbs = row["CarbsPerServing"] * serving
        fat = row["FatPerServing"] * serving

        calorie_diff = abs(calories - meal_calories) / meal_calories
        protein_diff = abs(protein - meal_protein) / meal_protein
        carb_diff = abs(carbs - meal_carbs) / meal_carbs
        fat_diff = abs(fat - meal_fat) / meal_fat

        distance = (
            CALORIE_WEIGHT * calorie_diff +
            PROTEIN_WEIGHT * protein_diff +
            CARB_WEIGHT * carb_diff +
            FAT_WEIGHT * fat_diff
        )

        if distance < best_distance:

            best_distance = distance
            best_serving = serving

            best_calories = calories
            best_protein = protein
            best_carbs = carbs
            best_fat = fat

    recommendations.append({
    "RecipeId": row["RecipeId"],
    "Name": row["Name"],
    "BestServing": best_serving,
    "RecommendedCalories": round(best_calories, 2),
    "RecommendedProtein": round(best_protein, 2),
    "RecommendedCarbs": round(best_carbs, 2),
    "RecommendedFat": round(best_fat, 2),
    "Distance": round(best_distance, 4),
    "NutritionScore": round(
        np.exp(-best_distance),
        4
    )
    })

recommendation_df = pd.DataFrame(recommendations)

recommendation_df.head()

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore
0,179272.0,Meatballs,4,534.70,40.50,6.80,37.70,0.3450,0.7082
1,538785.0,Spinach Artichoke Mac and Cheese,9,564.75,26.46,56.61,26.91,0.2560,0.7742
2,419924.0,Curried Brussels Sprouts Casserole,4,52.00,4.10,7.30,1.40,0.9002,0.4065
3,279131.0,Crock Pot Button Bone BBQ Pork,2,455.00,52.20,15.10,19.25,0.3153,0.7296
4,247849.0,Chocolate Peanut Butter Protein Oatmeal,1,343.70,14.50,33.40,18.70,0.4600,0.6313


In [18]:
# Top Recommendations

top_10 = (
    recommendation_df
    .sort_values("Distance")
    .head(10)
    .reset_index(drop=True)
)

top_10

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore
0,526321.0,Meatloaf With Maple Bourbon BBQ Sauce,3,513.30,42.45,37.95,21.08,0.1480,0.8625
1,161723.0,Hot Turkey Sandwiches,4,489.70,23.60,56.80,18.40,0.2304,0.7942
2,163707.0,Fish Parcels With Vegetables,3,441.98,34.80,71.40,4.50,0.2407,0.7860
3,84578.0,Roast Pork Loin with Pasta,5,648.00,31.67,62.58,28.17,0.2470,0.7812
4,538785.0,Spinach Artichoke Mac and Cheese,9,564.75,26.46,56.61,26.91,0.2560,0.7742
5,413891.0,"Cheese, Ham and Vegetable Pasta Bake",4,629.50,31.40,59.40,30.50,0.2683,0.7646
6,294255.0,Pineapple-Glazed Chicken,3,575.03,34.35,36.53,31.58,0.2740,0.7604
7,203767.0,Oysters Osaka,4,334.40,37.90,21.20,9.20,0.3028,0.7387
8,343653.0,Tofu Breakfast Burrito,4,421.00,23.20,63.50,9.50,0.3031,0.7385
9,280582.0,Chicken Charlena,10,383.50,28.00,34.30,13.50,0.3039,0.7379


### Evaluation Findings

Recipes with lower distance scores provide a nutritional profile that is closer to the user's target meal requirements.

The selected serving size represents the serving quantity that minimizes the weighted nutritional distance.

These recipes constitute the initial recommendation set generated by the system.

### Recommendation Quality Analysis

The nutritional profile of each recommended recipe can now be compared against the target meal requirements.

Target Meal:

- Calories: 550 kcal
- Protein: 40 g
- Carbohydrates: 62.5 g
- Fat: 16.25 g

Recipes with lower distance scores should provide nutritional values that are closer to these targets.

In [19]:
# Target Meal Summary

target_summary = pd.DataFrame({
    "Metric": ["Calories", "Protein", "Carbs", "Fat"],
    "Target": [
        meal_calories,
        meal_protein,
        meal_carbs,
        meal_fat
    ]
})

target_summary

,Metric,Target
0,Calories,550.00
1,Protein,40.00
2,Carbs,62.50
3,Fat,16.25


### Best Recommendation

The highest-ranked recipe represents the closest nutritional match identified by the recommendation engine.

The nutritional profile of this recommendation is compared against the target meal requirements below.

In [20]:
best_recipe = top_10.iloc[0]

best_recipe

RecipeId                                            526321.0
Name                   Meatloaf With Maple Bourbon BBQ Sauce
BestServing                                                3
RecommendedCalories                                    513.3
RecommendedProtein                                     42.45
RecommendedCarbs                                       37.95
RecommendedFat                                         21.08
Distance                                               0.148
NutritionScore                                        0.8625
Name: 0, dtype: object

In [21]:
comparison_df = pd.DataFrame({
    "Metric": ["Calories", "Protein", "Carbs", "Fat"],
    "Target": [
        meal_calories,
        meal_protein,
        meal_carbs,
        meal_fat
    ],
    "Recommended": [
        best_recipe["RecommendedCalories"],
        best_recipe["RecommendedProtein"],
        best_recipe["RecommendedCarbs"],
        best_recipe["RecommendedFat"]
    ]
})

comparison_df["Difference"] = (
    comparison_df["Recommended"]
    - comparison_df["Target"]
).round(2)

comparison_df

,Metric,Target,Recommended,Difference
0,Calories,550.00,513.30,-36.70
1,Protein,40.00,42.45,2.45
2,Carbs,62.50,37.95,-24.55
3,Fat,16.25,21.08,4.83


## Discussion

The highest-ranked recommendation provides a nutritional profile that is reasonably close to the target meal requirements.

Small deviations are expected because recipes are constrained by:

- Available nutritional profiles
- Whole-number serving sizes
- Original recipe serving counts

Despite these constraints, the recommendation engine is able to identify recipes that closely align with the user's nutritional goals while maintaining realistic serving quantities.

## Conclusion

A personalized recipe recommendation engine was developed using nutritional information and dietary preferences.

The system:

- Filters recipes according to dietary requirements.
- Converts daily nutritional goals into meal-level targets.
- Evaluates valid serving quantities for each recipe.
- Computes a weighted nutritional distance score.
- Ranks recipes according to their similarity to the target meal profile.

The resulting recommendations provide nutritionally relevant meal suggestions while maintaining realistic serving sizes.

The next stage of the project will focus on constructing complete daily meal plans by sequentially selecting recipes and continuously updating the user's remaining nutritional targets.